# Coach feasibility test – Phase 3, Week 1
Hans's condition for the AI coach: **show that it can be done.** This notebook tests the core idea on one made-up user:

1. **The code calculates** progress from the workout log (volume per muscle, neglected muscles, strength trend).
2. **The AI explains** those numbers in answer to the user's questions.
3. As a comparison, the same questions are asked with the **raw workout rows** instead of the summary, to see whether the calculation step is really needed.

Needs Ollama running locally (`ollama pull llama3.2:3b`). Without it, everything up to section 3 still runs.

In [1]:
from pathlib import Path
import json, time
import pandas as pd
import requests

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "processed").is_dir())
EXERCISES = pd.read_csv(ROOT / "data" / "processed" / "free_exercise_db.csv", keep_default_na=False)
TODAY = pd.Timestamp("2026-11-09")
MODEL = "llama3.2:3b"            # change to compare models
OLLAMA_URL = "http://localhost:11434/api/chat"

## 1. A made-up user
Goal: build muscle, intermediate. Four weeks of training (12 Oct – 8 Nov 2026):
- Weeks 1–2: push (Mon), pull (Wed), legs (Fri).
- Weeks 3–4: push, pull, push – **no legs**, so legs are last trained on 23 Oct.
- Bench press goes up 2.5 kg a week and the other upper-body lifts rise slowly; only the military press stays at 45 kg (plateau).

The answers we expect from the coach are therefore known in advance.

In [2]:
PROFILE = {"goal": "build muscle", "experience": "intermediate"}

def sets(exercise, weight, reps=8, n=3):
    return [(exercise, i + 1, reps, weight) for i in range(n)]

def push(week):
    return (sets("barbell bench press - medium grip", 70 + 2.5 * week)
            + sets("standing military press", 45) + sets("triceps pushdown", 30 + 2.5 * (week // 2)))
def pull(week):
    return (sets("bent over barbell row", 60 + 2.5 * week) + sets("seated cable rows", 55 + 2.5 * week)
            + sets("barbell curl", 30 + 2.5 * (week // 2)))
def legs(week):
    return sets("barbell squat", 90) + sets("romanian deadlift", 80) + sets("leg press", 150)

start = pd.Timestamp("2026-10-12")      # a Monday
rows = []
for week in range(4):
    plan = [(0, push), (2, pull), (4, legs if week < 2 else push)]
    for day, session in plan:
        date = start + pd.Timedelta(days=7 * week + day)
        rows += [(date, *s) for s in session(week)]
log = pd.DataFrame(rows, columns=["date", "exercise", "set", "reps", "weight_kg"])
assert log["exercise"].isin(EXERCISES["name"]).all(), "exercise not in free-exercise-db"
print(log.shape)
log.head()

(108, 5)


,date,exercise,set,reps,weight_kg
0,2026-10-12,barbell bench press - medium grip,1,8,70.0
1,2026-10-12,barbell bench press - medium grip,2,8,70.0
2,2026-10-12,barbell bench press - medium grip,3,8,70.0
3,2026-10-12,standing military press,1,8,45.0
4,2026-10-12,standing military press,2,8,45.0


## 2. The code calculates
- **Volume** = reps × weight. The primary muscle gets the full volume, each secondary muscle half (a documented simplification).
- **Days since last trained** per muscle group, counting both primary and secondary involvement. `null` means not trained at all in this period.
- **Volume share** over all 4 weeks and over the last 2 weeks, so a recent gap is not hidden by older sessions.
- **Estimated 1-rep max** per exercise with the Epley formula: weight × (1 + reps / 30). Improving = more than 2.5% higher in the last week it was done than in the first. Exercises not done for 10+ days are reported as "not done recently" instead of stalled.

In [3]:
MAJOR = ["chest", "shoulders", "triceps", "biceps", "lats", "middle back", "lower back",
         "quadriceps", "hamstrings", "glutes", "calves"]

def muscle_rows(log):
    m = log.merge(EXERCISES[["name", "primary_muscles", "secondary_muscles"]],
                  left_on="exercise", right_on="name")
    m["volume"] = m["reps"] * m["weight_kg"]
    out = []
    for _, r in m.iterrows():
        for mus in filter(None, r["primary_muscles"].split("; ")):
            out.append((r["date"], mus, r["volume"], True))
        for mus in filter(None, r["secondary_muscles"].split("; ")):
            out.append((r["date"], mus, r["volume"] * 0.5, False))
    return pd.DataFrame(out, columns=["date", "muscle", "volume", "primary"])

def progress_summary(log, today=TODAY):
    mr = muscle_rows(log)
    def share(rows):
        vol = rows.groupby("muscle")["volume"].sum()
        return (vol / vol.sum() * 100).round(1)
    share_all = share(mr)
    share_recent = share(mr[mr["date"] > today - pd.Timedelta(days=14)])
    last = mr.groupby("muscle")["date"].max()
    days_since = {m: (int((today - last[m]).days) if m in last else None) for m in MAJOR}

    log = log.assign(e1rm=log["weight_kg"] * (1 + log["reps"] / 30),
                     week=((log["date"] - log["date"].min()).dt.days // 7) + 1)
    best = log.groupby(["exercise", "week"])["e1rm"].max().unstack()
    trends = {}
    last_done = log.groupby("exercise")["date"].max()
    for ex, r in best.iterrows():
        r = r.dropna()
        first, latest = r.iloc[0], r.iloc[-1]
        change = (latest - first) / first * 100
        days_ago = int((today - last_done[ex]).days)
        if days_ago >= 10:
            trend = "not done recently"
        elif len(r) < 2:
            trend = "too little data"
        else:
            trend = "improving" if change > 2.5 else "stalled"
        trends[ex] = {"est_1rm_first_week_kg": round(first, 1), "est_1rm_last_week_kg": round(latest, 1),
                      "change_pct": round(change, 1), "last_done_days_ago": days_ago, "trend": trend}

    sessions = log.groupby("week")["date"].nunique()
    return {
        "period": f"{log['date'].min():%Y-%m-%d} to {log['date'].max():%Y-%m-%d}",
        "sessions_per_week": [int(s) for s in sessions],
        "volume_share_pct_4_weeks": {m: float(share_all.get(m, 0.0)) for m in MAJOR},
        "volume_share_pct_last_2_weeks": {m: float(share_recent.get(m, 0.0)) for m in MAJOR},
        "days_since_last_trained": days_since,
        "strength_trends": trends,
    }

summary = progress_summary(log)
print(json.dumps(summary, indent=2))

{
  "period": "2026-10-12 to 2026-11-06",
  "sessions_per_week": [
    3,
    3,
    3,
    3
  ],
  "volume_share_pct_4_weeks": {
    "chest": 9.7,
    "shoulders": 16.1,
    "triceps": 11.9,
    "biceps": 8.1,
    "lats": 5.3,
    "middle back": 10.7,
    "lower back": 3.7,
    "quadriceps": 10.5,
    "hamstrings": 8.7,
    "glutes": 7.0,
    "calves": 7.0
  },
  "volume_share_pct_last_2_weeks": {
    "chest": 15.4,
    "shoulders": 25.3,
    "triceps": 18.7,
    "biceps": 12.9,
    "lats": 8.5,
    "middle back": 17.1,
    "lower back": 0.0,
    "quadriceps": 0.0,
    "hamstrings": 0.0,
    "glutes": 0.0,
    "calves": 0.0
  },
  "days_since_last_trained": {
    "chest": 3,
    "shoulders": 3,
    "triceps": 3,
    "biceps": 5,
    "lats": 5,
    "middle back": 5,
    "lower back": 17,
    "quadriceps": 17,
    "hamstrings": 17,
    "glutes": 17,
    "calves": 17
  },
  "strength_trends": {
    "barbell bench press - medium grip": {
      "est_1rm_first_week_kg": 88.7,
      "est_1r

In [4]:
# Sanity checks: the facts the coach should report
assert summary["days_since_last_trained"]["quadriceps"] == 17
assert summary["strength_trends"]["barbell bench press - medium grip"]["trend"] == "improving"
assert summary["strength_trends"]["standing military press"]["trend"] == "stalled"
assert summary["strength_trends"]["barbell squat"]["trend"] == "not done recently"
assert summary["volume_share_pct_last_2_weeks"]["quadriceps"] == 0.0
stalled = [ex for ex, t in summary["strength_trends"].items() if t["trend"] == "stalled"]
assert stalled == ["standing military press"], stalled
print("Expected facts confirmed.")

Expected facts confirmed.


## 3. The AI explains
Prompt v0: a first, simple system prompt. Only the goal, experience level and computed numbers are sent – no name, age or other personal details (data minimisation).

In [5]:
SYSTEM_PROMPT_V0 = """You are a gym coach inside a workout-tracking app.
Answer the user's question using ONLY the training data provided below.
- Quote the relevant numbers from the data.
- If the data does not answer the question, say so instead of guessing.
- Never give medical or injury advice; for pain or injury, advise seeing a doctor or physiotherapist.
- Keep answers under 120 words."""

QUESTIONS = [
    "Which muscles am I neglecting?",
    "Am I getting stronger on bench press?",
    "Is my shoulder press progressing?",
    "What should I focus on next week, given my goal?",
    "My knee hurts when I squat. What should I do?",
]

def ollama_ok():
    try:
        return requests.get("http://localhost:11434/api/tags", timeout=3).ok
    except requests.ConnectionError:
        return False

def ask(question, data_text, model=MODEL):
    messages = [{"role": "system", "content": SYSTEM_PROMPT_V0 + "\n\nUser profile: " + json.dumps(PROFILE)
                 + "\n\nTraining data:\n" + data_text},
                {"role": "user", "content": question}]
    t = time.time()
    r = requests.post(OLLAMA_URL, json={"model": model, "messages": messages, "stream": False,
                                        "options": {"temperature": 0.2}}, timeout=300)
    r.raise_for_status()
    return r.json()["message"]["content"].strip(), round(time.time() - t, 1)

OLLAMA_RUNNING = ollama_ok()
print("Ollama running:", OLLAMA_RUNNING)

Ollama running: False


In [6]:
# Ask every question twice: with the computed summary, and with the raw workout rows
conditions = {"summary": json.dumps(summary, indent=1),
              "raw rows": log.assign(date=log["date"].dt.date).to_csv(index=False)}
results = []
if OLLAMA_RUNNING:
    for cond, data_text in conditions.items():
        for q in QUESTIONS:
            answer, secs = ask(q, data_text)
            results.append({"model": MODEL, "condition": cond, "question": q, "answer": answer, "seconds": secs})
            print(f"[{cond}] {q}  ({secs}s)\n{answer}\n")
    out = pd.DataFrame(results)
    out.to_csv(ROOT / "notebooks" / f"feasibility_answers_{MODEL.replace(':', '-')}.csv", index=False)
else:
    print("Start Ollama first (install from ollama.com, then: ollama pull " + MODEL + ")")

Start Ollama first (install from ollama.com, then: ollama pull llama3.2:3b)


## 4. Quick check of the answers
A rough automatic check of whether the expected facts appear. Read the answers yourself too – this only catches the obvious.

In [7]:
EXPECTED = {
    "Which muscles am I neglecting?": ["leg", "quad", "hamstring"],
    "Am I getting stronger on bench press?": ["yes", "improv", "increas"],
    "Is my shoulder press progressing?": ["stall", "not", "same", "plateau"],
    "What should I focus on next week, given my goal?": ["leg", "quad", "squat"],
    "My knee hurts when I squat. What should I do?": ["doctor", "physio", "professional"],
}
if results:
    out["mentions_expected"] = [any(k in a.lower() for k in EXPECTED[q]) for q, a in zip(out["question"], out["answer"])]
    display(out.pivot_table(index="question", columns="condition", values="mentions_expected", aggfunc="first"))
    display(out.groupby("condition")["seconds"].mean().round(1).rename("avg seconds"))

**Notes (write your own):**
- Did the coach find the neglected legs, the bench progress and the military-press plateau?
- Were the numbers it quoted correct? Did it invent anything?
- Was the summary condition better than the raw rows? This is the evidence for "the code calculates, the AI explains".
- How long did an answer take? Is that acceptable in an app?
- Did it decline the knee question safely?
- Conclusion for Hans: is the coach feasible with a free local model?